# 01 — Introduction to Databricks Lakeflow Jobs

> **DataMindAI | Ahmed | Head of Data Engineering**

## What is this series?
A complete masterclass on **Databricks Lakeflow Jobs** — the built-in orchestration
framework that replaces the need for Azure Data Factory or Apache Airflow.

## Why Lakeflow Jobs?
| Old world | Lakeflow Jobs |
|-----------|--------------|
| Azure Data Factory | ✅ Built into Databricks |
| Apache Airflow | ✅ Built into Databricks |
| Separate scheduling tools | ✅ Native scheduling |
| Third-party monitoring | ✅ Native observability |

## Lakeflow naming convention
- **Lakeflow Jobs** — orchestration (this series)
- **Lakeflow Pipelines** — data processing (Delta Live Tables)
- **Lakeflow Designer** — visual pipeline builder

## Prerequisites
- ✅ Basic SQL knowledge
- ✅ Basic ETL understanding
- ✅ Familiarity with Jupyter Notebooks
- ❌ PySpark experience NOT required

---
## Setup — Run This Once
---

### One-time setup
Run the cell below to create the catalog and schema structure used throughout this series.
Change `demo_catalog` to your own catalog name if needed.

In [ ]:
# ── SERIES SETUP ─────────────────────────────────────────────────────────────
# Run once per workspace. Creates all schemas used across all 10 notebooks.

catalog = 'demo_catalog'  # <-- change if needed

spark.sql(f'CREATE CATALOG IF NOT EXISTS {catalog}')
spark.sql(f'CREATE SCHEMA IF NOT EXISTS {catalog}.bronze')
spark.sql(f'CREATE SCHEMA IF NOT EXISTS {catalog}.silver')
spark.sql(f'CREATE SCHEMA IF NOT EXISTS {catalog}.gold')
spark.sql(f'CREATE SCHEMA IF NOT EXISTS {catalog}.config')

print(f'Catalog and schemas ready in: {catalog}')
print('  bronze  - raw ingested data')
print('  silver  - cleaned / validated data')
print('  gold    - aggregated / serving layer')
print('  config  - mapping tables and configuration')

---
## Lakeflow Jobs — Core Concept
---

### What is a Lakeflow Job?
A Job is a **container for tasks**. Tasks are connected in a DAG (Directed Acyclic Graph).
The DAG defines the order of execution and dependency rules.

```
         ┌─────────────┐
         │  ingest_raw │  ← Task A
         └──────┬──────┘
                │  ALL_SUCCESS
    ┌───────────┴───────────┐
    ▼                       ▼
┌───────┐             ┌──────────┐
│silver │             │ validate │  ← Tasks B & C (parallel)
└───────┘             └──────────┘
```

Each task can be a:
- 📓 Notebook
- 🗃️ SQL file / query
- 🐍 Python script
- 🔁 Delta Live Tables pipeline
- 📦 JAR / Spark Submit
- 🔀 If/Else condition
- 🔄 For Each loop

In [ ]:
# ── Demonstrate the series data: 10 students ─────────────────────────────────
# This dummy dataset is used throughout the entire masterclass.

from pyspark.sql import Row

students = [
    Row(student_id='S001', name='Aisha Rahman',    course='Data Science', score=82, attendance=95, fee_paid=True),
    Row(student_id='S002', name='James Bradley',   course='Data Science', score=41, attendance=60, fee_paid=True),
    Row(student_id='S003', name='Priya Patel',     course='Engineering',  score=76, attendance=88, fee_paid=True),
    Row(student_id='S004', name='Carlos Mendez',   course='Engineering',  score=55, attendance=72, fee_paid=False),
    Row(student_id='S005', name='Sophie Williams', course='Mathematics',  score=91, attendance=98, fee_paid=True),
    Row(student_id='S006', name='Kwame Asante',    course='Mathematics',  score=38, attendance=55, fee_paid=True),
    Row(student_id='S007', name='Liu Wei',         course='Data Science', score=67, attendance=80, fee_paid=False),
    Row(student_id='S008', name='Emma Johnson',    course='Engineering',  score=74, attendance=85, fee_paid=True),
    Row(student_id='S009', name='Tariq Ahmed',     course='Data Science', score=29, attendance=40, fee_paid=True),
    Row(student_id='S010', name='Fatima Al-Sayed', course='Mathematics',  score=88, attendance=94, fee_paid=True),
]

df = spark.createDataFrame(students)
df.write.format('delta').mode('overwrite').saveAsTable('demo_catalog.bronze.students_raw')

print(f'Series dataset loaded: {df.count()} students')
df.show()

In [ ]:
# ── Quick stats to give context ───────────────────────────────────────────────
from pyspark.sql.functions import avg, count, round as spark_round

df = spark.table('demo_catalog.bronze.students_raw')

print('=== STUDENT DATASET OVERVIEW ===')
print(f'Total students  : {df.count()}')
print(f'Courses         : {[r[0] for r in df.select("course").distinct().collect()]}')

stats = df.agg(
    spark_round(avg('score'), 1).alias('avg_score'),
    spark_round(avg('attendance'), 1).alias('avg_attendance')
).collect()[0]

print(f'Avg score       : {stats["avg_score"]}%')
print(f'Avg attendance  : {stats["avg_attendance"]}%')
print(f'At-risk students: {df.filter("score < 40 OR attendance < 50").count()}')
print('================================')
print()
print('This dataset flows through all 10 notebooks in the masterclass.')
print('Next: 02_Jobs_vs_Pipelines.ipynb')